# **Lab:** Security in Computer Networks

**Student name:** write your name here

### **Topics**
- Confidentiality
- Public Key Cryptography (RSA)
- Digital Signatures

In [1]:
# Install cryptography library (only first time)
# This library provides tools for encryption, decryption, signatures, etc.
%pip install cryptography

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import RSA module for asymmetric cryptography
from cryptography.hazmat.primitives.asymmetric import rsa

# Import serialization tools (used for saving/loading keys, not used deeply here)
from cryptography.hazmat.primitives import serialization

# Import padding schemes (needed for secure encryption)
from cryptography.hazmat.primitives.asymmetric import padding

# Import hashing algorithms (SHA256, etc.)
from cryptography.hazmat.primitives import hashes

## **🔐 Part 1:** RSA (Public Key Cryptography)

- Public key → shared
- Private key → secret
- Encrypt with public key, decrypt with private key


In [3]:
# Generate PRIVATE key
# public_exponent=65537 is a standard safe value used in RSA
# key_size=2048 means 2048-bit key (secure for most purposes)
private_key = rsa.generate_private_key(
    public_exponent=65537,   # mathematical parameter used in RSA
    key_size=2048            # size of the key in bits (larger = more secure but slower)
)

# Generate PUBLIC key from private key
# In RSA, both keys are mathematically linked
public_key = private_key.public_key()

print("Keys generated successfully")

Keys generated successfully


In [4]:
# Export public key to PEM
pem_pub = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
).decode()

pem_pub

'-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEA0MwjVCVRq/MH/LrEQX3b\nYKurr608hAG9Hkq0Z0o/ttp8I6Zan95Up+t7PVdLVByS8kD5PkJ9BVlrg6FL8+Fs\nkTXw7XMZaICFxV/DgMdIH9BhM4B1brgRsgvLQeW2zcSiMbWxMlgI06IR3aYfMDiG\ng/2YdULW7G9SI8Dof3WStIcPesQtOrKrmEbWhk8OxgNngcP1m8mPkVG3Wz4HLA0w\npfWurWTBpyLP6AWcMgm99WVcA7WE7CFJDEPLieJnmid3x8gXMymLtAc39rHFIYoa\nUlmDl2oCB+tnHEWPZGN+6VJQaU0XfSCmuI0xTJzbqbxCvJYJ0qiFocoPyNbbZzI2\n7QIDAQAB\n-----END PUBLIC KEY-----\n'

In [5]:
# Convert received PEM to an object
received_public_key = serialization.load_pem_public_key(pem_pub.encode())

received_public_key

In [6]:
import base64

# Define message to encrypt
# .encode() converts string → bytes (required for crypto functions)
message = "Hello secure world".encode()

# Encrypt using PUBLIC key
encrypted = received_public_key.encrypt(
    message,  # data to encrypt (must be bytes)
    padding.OAEP(  # OAEP = Optimal Asymmetric Encryption Padding (secure padding)
        mgf=padding.MGF1(algorithm=hashes.SHA256()),  # MGF1 = Mask Generation Function
        algorithm=hashes.SHA256(),  # hash algorithm used internally
        label=None  # optional label (not needed here)
    )
)

print("Encrypted (Python bytes object):", encrypted) # This will show as a bytes object (not human-readable). ASCIIs are not suitable for binary data, so we will convert it to hex and base64 for display.
print("Encrypted (hex):", encrypted.hex())
print("Encrypted (base64):", base64.b64encode(encrypted).decode()) # Standard way to represent binary data as text

Encrypted (Python bytes object): b'\x88;\xa7\x9b\x94\x02\x031\xba\x97y*\x84uJ\xa2\xd5Cw\xa2\t\x1dj\xea\x8b\x19p\xcfqZ\x82\xc7\x99<0\xe8\x07\xc0\xb7f\xafpK\xe3jN\xbd\x1c<\x85\x13\xdc\x91\x9aW\xfbv\xc7\xae\xc4\xb7\xced\x7f\xc1F\xbb\xff\x12\xb1K\xadVk[\t\xf6\xbfEZ\xe4\xb4\xb3\x11\xed2\x1e8\xcf\x00ze\xf8\xe6M\xea\xfc\x84\xd4\x92\x81\xa4\xbe\xd2}{\xe3\x84\x84z\xc4\xea5K\x90\xc9Gy\x97\xe3B\n\xdd\xf3\x14\x84\x87L150\xccq\xcaq!\xfd-\x95\x06\x18\xd7\x0c\xba`\x92\x14\x18,d\xa7e1\x13\x1aI\x7f\xc8\x97\xce\xdc\x15\xd7\xf5\xc8\xdfq6A9\xbf\x80\xbfn\xae\xb6\xfe\x82\xdci\xfbE5\xc8\xb5S\x00\xa5?\xa7/ii\xf9\x9e\xfdC\xda)\xc9\x82\x96\x97[\x84\xf6u\xed\xb1\xd5\x1f\x0f_\x97\xab\xdb\xf2\x10m\x08y0\xedM\xe6\x1b\xbbk\xb0\xb9\x02\x93W\xb3\x17&\x07\xf2g\xf4\t\xbd\xa2\x19\x0fz\xcb\x9d\xdc\xc0\xe9\xe5\xeb\x84\x19\xcc'
Encrypted (hex): 883ba79b94020331ba97792a84754aa2d54377a2091d6aea8b1970cf715a82c7993c30e807c0b766af704be36a4ebd1c3c8513dc919a57fb76c7aec4b7ce647fc146bbff12b14bad566b5b09f6bf455ae4b4b311ed321e38cf007a

In [7]:
# Decrypt using PRIVATE key
decrypted = private_key.decrypt(
    encrypted,  # encrypted data
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Convert bytes → string using .decode()
print("Decrypted:", decrypted.decode())

Decrypted: Hello secure world


When you:

1. Take a message  
2. Encrypt it with the **receiver’s public key**  
3. Encode it in **Base64**

You already achieved **confidentiality**:  
Only the holder of the **private key** can decrypt it.

> ⚠️ **Base64 is NOT security**, it is just an encoding format to represent binary data as text.

The real problem (without certificates)

The key question is:

> ❗ How do you know that the **public key** actually belongs to the right person?

Without certificates, this can happen:

- An attacker gives you **their public key**
- You encrypt the message using that key
- The attacker decrypts it with their private key
- Then re-encrypts it using the real recipient’s key

*Result:* your communication was intercepted without you noticing.

This is known as a **Man-in-the-Middle** attack.

So… what are certificates for?

A certificate (e.g., `X.509`) is used to:

**Bind:**
- A **public key**
- To a **real identity** (person, server, organization)

**It also includes:**
- A trusted signature (from a Certificate Authority)
- Owner information
- Expiration/validity dates

Who guarantees this?

A **Certificate Authority (CA)** such as:

- Let's Encrypt  
- DigiCert  

These entities sign certificates and essentially say:

> “Yes, this public key belongs to this entity.”

## **✍️ Part 2:** Digital Signatures

- Sign with PRIVATE key
- Verify with PUBLIC key


In [9]:
# Create digital signature
signature = private_key.sign(
    message,  # original message
    padding.PSS(  # Probabilistic Signature Scheme (secure signature padding)
        mgf=padding.MGF1(hashes.SHA256()),  # mask generation function
        salt_length=padding.PSS.MAX_LENGTH  # random salt for security
    ),
    hashes.SHA256()  # hash algorithm used before signing
)

print("Signature created")

Signature created


In [10]:
# Verify signature
try:
    public_key.verify(
        signature,  # signature to verify
        message,    # original message
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Signature VALID")
except:
    print("Signature INVALID")

Signature VALID


How it fits together?

In protocols like **Transport Layer Security (TLS)**:

1. The server sends its certificate  
2. The client verifies:
   - The CA signature  
   - The domain name  
3. **Only then** the public key is trusted  
4. The key is used for encryption

## **🌐 Part 3:** TLS Handshake Simulation

Real TLS:
1. Client gets server public key
2. Client creates session key
3. Client encrypts session key with public key
4. Server decrypts
5. Both use symmetric encryption


In [11]:
# Import symmetric encryption tool
from cryptography.fernet import Fernet

In [12]:
# Client generates symmetric session key
# This key will be used for fast encryption (like AES)
session_key = Fernet.generate_key()

# Create cipher object for encryption/decryption
cipher_client = Fernet(session_key)

print("Session key created")

Session key created


In [13]:
# Client encrypts session key using server PUBLIC key
encrypted_session_key = public_key.encrypt(
    session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

In [14]:
# Server decrypts session key using PRIVATE key
decrypted_session_key = private_key.decrypt(
    encrypted_session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Server creates cipher with shared key
cipher_server = Fernet(decrypted_session_key)


In [15]:
# Secure communication
secure_message = cipher_client.encrypt(b"Hello TLS secure world")

print("Encrypted message:", secure_message)

print("Decrypted message:", cipher_server.decrypt(secure_message).decode())

Encrypted message: b'gAAAAABq-08DoIEfSXqqoF3QLK-HTbMzX9581U2q_OtqEpJIu9GZC5FaKB0YbnxOtA3vYdSCo6P8A5UR5eVLls4vVVUAtNQ1L5OjTtlGFbIUUxutJQYkwv8='
Decrypted message: Hello TLS secure world


In [ ]:
import base64

# Public Key Partner
compa_pem = """-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEA3XSV1Ua8Ebxd9mVq5rHE
cn8xbPumoetMUabh9AiL/vyCe6UBMEdtF9tWyq/ynJAYYeFTXy6d0Drwrp9P8i+1
XM20x9DQJW7Qo0G1q2ksckUek0YPdkqqZX5TXx9CjDD7vkdsi6t6/neaMq41UsMT
7MFK7DnAdQMou2B/ry0B0/BvghUb/TXBp/UIVP8bxkxiTmlyTHkTKaONUKvpbBHN
mEeCy6RTUjoNVMcGpG4MvjfyllMgvgDMIG7WftgUlpU0lQ3uKNd+Nim4voNwM+qK
c5LGIKBtBkV7gckmEMn5dMpLyWNYvr04AwDx8BVDM3qBkFvX6lrR56jWIJwxHodW
8QIDAQAB
-----END PUBLIC KEY-----"""

compa_public_key = serialization.load_pem_public_key(compa_pem.encode())

# Message  
mi_mensaje = "Hola, prueba de seguridad en redes".encode()

# Encrypt with partner's public key
mensaje_encriptado = compa_public_key.encrypt(
    mi_mensaje,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Sign with own private key
mi_firma = private_key.sign(
    mi_mensaje,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

# 5. IMPRIMIR EN BASE64 PARA ENVIAR POR CHAT
print("ENVÍA ESTO A TU COMPAÑERO\n")
print("MENSAJE ENCRIPTADO (Base64):")
print(base64.b64encode(mensaje_encriptado).decode())
print("\nFIRMA DIGITAL (Base64):")
print(base64.b64encode(mi_firma).decode())

=== ENVÍA ESTO A TU COMPAÑERO ===

MENSAJE ENCRIPTADO (Base64):
qYKfBFlku33+NxLsufXkrzKgh0bZt9/FUr2vp2JcgsXBcFYBAyCMwv7Y2npSSMMVqml8Juei8h//+TYVo+iMtkE4MqQ1kJhTsQKKAtA8rrNVgzEUbmMDzr9Qh1ODf79apNFaiCrRXus/4JVQlnCw7Xy332ZnUwOg/+y3Yr1ibNUrHRZQvPSUApjXOwgAgUuC6/8AxL9ZnKnFeXGM5iMLwN1jRRMum5+tG2t7CDueRGJSUCSMF8wRDkOeWfir3yEdLLr/CiHPCkKdA4hXJR200uw05SLxF4qGNzRe+gQl0cyeQtM4N+n+WsnSHPIBeXev2nm/8Kl/vrI0z8JbTD6QqA==

FIRMA DIGITAL (Base64):
lKjIqshxGn2hEg1j2fmS2CakBuq9wC9+BCRF/r2+LfFxQtWH/ek2NxdcR+pAnVPpHdVrFasBh8vMaBk+4a0CMalJRFwZDhxpiQaPDArmBHYoJYf5mEk1SF7UYpyYQLenMtDho0YYkFvShpeoYHDI5hXcodhFZggaK967QKLlGKUMWinu1Y2DkWlk88BFaZhKLhh1/RbR9zU7H5BCsmkLbdrM2UiEoGiqa+2oiyqsePWD2Y6SKAdzvW0Kxv849+S/ESg5FRWm/QXKu+TzZICS0EXh1iQJZYWx62er7iqzYaCYweaVFtrRw1ssPLpFGKfWcNVtzILuUh0lFhut39IJ8Q==


In [17]:
# 1. PEGA AQUÍ LO QUE TE MANDÓ TU COMPAÑERO (EN BASE64)
encrypted_b64_recibido = "v6gih1Wakqlk/2zfAzzxhmA1SODRTwkBPtC6BSaOOwSk0+SmbNv2aFyPMHV1pcNpo4Cg5myww7tdGPzq01iNhuXc5VwMvB6z/zau+VgOc7QcYkCqnc2H3cw1GEHy3rDMvQQ8drYPR5XTsxrW1wDUgeBwJakutB1Tukyr1bsW5K1kq/NKMdtl88T1ANTD1fjGCdtNyGY6oGsJylM1RNT0iaCTBTjEX3WcboZPi9kEEWQuhIY5KmP6AOjMM1R8mkpC+2ABgiZCMrX20dFm/DdXHcEXRNWnxSeGkUHqmvLj8Or3nmJsHxyLIQYcykGfQHZcrX6XTk5jeTzmZysKS9528Q=="
signature_b64_recibida = "mTOne1W65F1mIMlrTuhXEVhE5Gr3GajL+5al+L2XUr22dQ/+UvikjkBHGtaNEfX863GYNyvESofvdKpqJstCiEB6t+6GxdXHVwj65LDTFAsAVEQEWvghyJeNYBlgmRHEFA36slvl8rr5StTN1qFSRsSpXDLF11ME5R5dPkndAPGmTcNHfIKzwNPPt8HAEsZEbYnpOejfL+3IsZYqQb2Z/KROY65VHHSu7dSN+rSPToAo+6+VGIKK51wMWLXPwfqzZc+WqvXhxp9OiLvbQSHRurcCt78ynsGSV9ALeMhQX59JZIhczS8ol4+PTmkpnmdTbsnesR8xEWtmCgT9o+aGZA=="

# Convertir de Base64 a bytes
encrypted_bytes = base64.b64decode(encrypted_b64_recibido)
signature_bytes = base64.b64decode(signature_b64_recibida)

# 2. DESENCRIPTAR CON TU PROPIA LLAVE PRIVADA
mensaje_desencriptado = private_key.decrypt(
    encrypted_bytes,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print("Mensaje recibido:", mensaje_desencriptado.decode())

# 3. VERIFICAR LA FIRMA CON LA LLAVE PÚBLICA DE TU COMPAÑERO
try:
    compa_public_key.verify(
        signature_bytes,
        mensaje_desencriptado,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Estado: Signature VALID ✔ (Mensaje aceptado)")
except Exception:
    print("Estado: Signature INVALID ❌ (Mensaje rechazado)")

Mensaje recibido: Confff, prueba de seguridad en redes
Estado: Signature VALID ✔ (Mensaje aceptado)


## **Assignment:** Secure Communication Challenge

### **Objective**
Apply all concepts from the lab:
- Confidentiality (encryption)
- Authentication (digital signatures)
- Integrity (verification)

### **Group Dynamics (Balanced Work)**

Each student must complete ALL roles:

1. **Sender**
2. **Receiver**

You will interact with at least **one other student**.

### **Task 1:** Secure Message Sending

Each student must:

1. Generate an RSA key pair
2. Share ONLY the **public key**
3. Create a message
4. Encrypt the message using the receiver’s public key
5. Sign the message using your private key

👉 Send to another student:
- Encrypted message
- Signature
- Your public key

### **Task 2:** Secure Message Receiving

Each student must:

1. Receive:
   - Encrypted message
   - Signature
   - Sender's public key

2. Decrypt the message using your private key  
3. Verify the signature using the sender’s public key  

✔ If valid → accept message  
❌ If invalid → reject message  

### **Time Limit**
- 20–25 minutes total

### **Evaluation (Checklist)**

| **Criteria** | **Score (10pts)** |
|--------|------|
| RSA encryption implemented | 1 pts |
| Digital signature implemented | 1 pts |
| Correctly decrypted & verified message | 4 pts |
| Explanation reflection questions | 4 pts |

### **Reflection Questions**

1. Why is encryption alone NOT enough for security?
Encryption only guarantees *confidentiality* (hiding the content), but it does not provide *authentication* or *integrity*. Without digital signatures or certificates, anyone with the public key could send a message, or an attacker could perform a Man-in-the-Middle attack by impersonating the sender's public key.  
2. What is the role of digital signatures?  
Digital signatures provide *authentication* (proving the message actually came from the holder of the private key), *integrity* (verifying the message was not altered in transit), and *non-repudiation*.
3. Why does TLS use a hybrid approach?  
Asymmetric cryptography (RSA) is computationally expensive and slow for large amounts of data, while symmetric cryptography (AES/Fernet) is fast. TLS uses RSA during the initial handshake to securely exchange a temporary symmetric session key, and then uses that symmetric key to encrypt the actual communication efficiently.

In [8]:
print(pem_pub)

-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEA0MwjVCVRq/MH/LrEQX3b
YKurr608hAG9Hkq0Z0o/ttp8I6Zan95Up+t7PVdLVByS8kD5PkJ9BVlrg6FL8+Fs
kTXw7XMZaICFxV/DgMdIH9BhM4B1brgRsgvLQeW2zcSiMbWxMlgI06IR3aYfMDiG
g/2YdULW7G9SI8Dof3WStIcPesQtOrKrmEbWhk8OxgNngcP1m8mPkVG3Wz4HLA0w
pfWurWTBpyLP6AWcMgm99WVcA7WE7CFJDEPLieJnmid3x8gXMymLtAc39rHFIYoa
UlmDl2oCB+tnHEWPZGN+6VJQaU0XfSCmuI0xTJzbqbxCvJYJ0qiFocoPyNbbZzI2
7QIDAQAB
-----END PUBLIC KEY-----

